# Part 1: Document Loaders in LangChain

In [68]:
from pathlib import Path
from langchain_community.document_loaders import (
    TextLoader,
    CSVLoader,
    PyPDFLoader,
    DirectoryLoader,
    WebBaseLoader,
)

In [69]:
DATA_DIR = Path("data")

## Task 1: Text Loader

In [70]:
txt_loader = TextLoader(
    str(DATA_DIR / "notes.txt"),
    encoding="utf-8"
)

txt_documents = txt_loader.load()

print("Number of documents loaded:", len(txt_documents))

Number of documents loaded: 1


In [71]:
txt_documents[0].page_content[:500]

'Personal Knowledge Assistant\n\nLangChain is a framework for building applications powered by large language models.\n\nA document loader is used to load information from different sources such as\ntext files, CSV files, PDF documents, websites, and directories.\n\nDocument loaders convert different types of data into LangChain Document objects.\n\nEach Document generally contains:\n1. page_content - the actual text\n2. metadata - information about the source document\n\nAfter loading documents, the data can'

In [72]:
txt_documents[0].metadata

{'source': 'data\\notes.txt'}

## TASK 2: CSV Loader

In [73]:
csv_loader = CSVLoader(
    str(DATA_DIR / "employees.csv")
)

In [74]:
csv_documents = csv_loader.load()

In [75]:
print(csv_documents[0].page_content)

name: Arun
department: Engineering
role: Software Engineer
experience: 2


In [76]:
print(csv_documents[0].metadata)

{'source': 'data\\employees.csv', 'row': 0}


## TASK 3: PDF Loader

In [77]:
pdf_loader = PyPDFLoader(
    str(DATA_DIR / "dummy_genai_document.pdf")
)
pdf_documents = pdf_loader.load()

In [78]:
print("Total pages:", len(pdf_documents))

Total pages: 1


In [79]:
pdf_documents[0].page_content[:500]

'Dummy GenAI Knowledge Document\nGenerative AI is a branch of artificial intelligence that can create new content such as text, code, images, and\nsummaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context\nprovided to them.\nRetrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from\nexternal documents before generating an answer. A typical RAG pipeline loads documents, splits them into\nchunks, creates embeddings, s'

In [80]:
pdf_documents[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-09-13T11:47:10+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-09-13T11:47:10+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'data\\dummy_genai_document.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}

## TASK 4 : Directory Loader

In [81]:
directory_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

In [82]:
txt_files = directory_loader.load()

In [83]:
csv_directory_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.csv",
    loader_cls=CSVLoader,
)

In [84]:
csv_files = csv_directory_loader.load()

print("CSV documents loaded:", len(csv_files))

CSV documents loaded: 5


In [85]:
pdf_directory_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

pdf_files = pdf_directory_loader.load()

print("PDF documents loaded:", len(pdf_files))

PDF documents loaded: 1


In [86]:
print("TXT:", len(txt_files), "document(s)")
print("CSV:", len(csv_files), "document(s)")
print("PDF:", len(pdf_files), "page(s)")

TXT: 1 document(s)
CSV: 5 document(s)
PDF: 1 page(s)


## TASK 5: WebBase Loader

In [87]:
url = "https://www.python.org/about/gettingstarted/"

In [88]:
web_loader = WebBaseLoader(url)

web_documents = web_loader.load()

In [89]:
len(web_documents)

1

In [90]:
web_documents[0].page_content[:500]

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPython For Beginners | Python.org\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNotice: This page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.\n\n\n\n\n\nSkip to content\n\n\n▼ Close\n                \n\n\nPython\n\n\nPSF\n\n\nDocs\n\n\nPyPI\n\n\nJobs\n\n\nCommunity\n\n\n\n▲ The Python Network\n                \n\n\n\n\n\n\n\n\n\nDonate\n\n≡ Menu\n\n\nSearch This Site\n\n\n                                    GO\n                              '

In [91]:
web_documents[0].metadata

{'source': 'https://www.python.org/about/gettingstarted/',
 'title': 'Python For Beginners | Python.org',
 'description': 'The official home of the Python Programming Language',
 'language': 'en'}

# Part 2: Text Splitters in Langchain

## TASK 6: Why Text Splitting is Required

1. Why larege documents cannot be directly passed to LLMs?
- Large documents can exceed an LLM's context window. 
- Even when a document fits, sending the entire document for every query is inefficient, expensive, and can reduce the model's ability to focus on the relevant information.

2. What problems does chunking solve in GenAI System?
- Breaks large documents into smaller manageable pieces.
- Helps stay within the LLM context window.
- Makes retrieval more precise.
- Reduces unnecessary tokens and cost.

## TASK 7: Character Based Text Splitter

In [92]:
from langchain_text_splitters import CharacterTextSplitter

In [93]:
documents = txt_documents

In [94]:
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=50
)

In [95]:
chunks = text_splitter.split_documents(documents)

In [96]:
len(chunks)

4

In [97]:
chunks[0].page_content

'Personal Knowledge Assistant\nLangChain is a framework for building applications powered by large language models.\nA document loader is used to load information from different sources such as'

In [98]:
chunks[0].metadata

{'source': 'data\\notes.txt'}

## TASK 8: Structure Based Text Splitter

In [99]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [100]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)


In [101]:
recursive_chunks = recursive_splitter.split_documents(documents)

In [102]:
len(recursive_chunks)

6

In [103]:
recursive_chunks[0].page_content

'Personal Knowledge Assistant\n\nLangChain is a framework for building applications powered by large language models.'

In [104]:
recursive_chunks[0].metadata

{'source': 'data\\notes.txt'}

In [105]:
# Comparison

In [106]:
print("CharacterTextSplitter chunks:",len(chunks))


CharacterTextSplitter chunks: 4


In [107]:
print("RecursiveCharacterTextSplitter chunks:",len(recursive_chunks))

RecursiveCharacterTextSplitter chunks: 6


## TASK 9: Document Structure Based Splitting

In [108]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter
)

In [109]:
markdown_text = """
# Python

Python is a popular programming language.

## Variables

Variables are used to store data.

## Functions

Functions are reusable blocks of code.

# LangChain

LangChain is used to build LLM applications.

## Document Loaders

Document loaders load data from different sources.

## Text Splitters

Text splitters divide large documents into smaller chunks.
"""

In [110]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

In [111]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

markdown_chunks = markdown_splitter.split_text(markdown_text)

In [112]:
for i, chunk in enumerate(markdown_chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)



Chunk 1:
Python is a popular programming language.
Metadata: {'Header 1': 'Python'}

Chunk 2:
Variables are used to store data.
Metadata: {'Header 1': 'Python', 'Header 2': 'Variables'}

Chunk 3:
Functions are reusable blocks of code.
Metadata: {'Header 1': 'Python', 'Header 2': 'Functions'}

Chunk 4:
LangChain is used to build LLM applications.
Metadata: {'Header 1': 'LangChain'}

Chunk 5:
Document loaders load data from different sources.
Metadata: {'Header 1': 'LangChain', 'Header 2': 'Document Loaders'}

Chunk 6:
Text splitters divide large documents into smaller chunks.
Metadata: {'Header 1': 'LangChain', 'Header 2': 'Text Splitters'}


In [113]:
pdf_loader = PyPDFLoader("data/dummy_genai_document.pdf")

pdf_documents = pdf_loader.load()

In [114]:
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [115]:
pdf_chunks = pdf_splitter.split_documents(pdf_documents)

In [116]:
for i, chunk in enumerate(pdf_chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk.page_content[:300])

    print("Metadata:", chunk.metadata)


Chunk 1:
Dummy GenAI Knowledge Document
Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and
summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context
provided to them.
Retrieval-Augmented Generation (RAG) im
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-13T11:47:10+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-13T11:47:10+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/dummy_genai_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}

Chunk 2:
chunks, creates embeddings, stores the embeddings in a vector store, retrieves relevant chunks, and sends
the retrieved context to the language model.
Prompt templates make LLM applications easier to maintain. Instead of hard-coding every question, a
template can contain placeholders such as {ques

## TASK 10: Semantic  Meaning Based Splitting

1. What is semantic chunking?

Semantic chunking means splitting a document based on meaning rather than only character count.

2. How do embeddings help?

Embeddings convert text into numerical vectors representing its semantic meaning.

# Part 3: Mini Integration Task

## TASK 11: Mini Integration Task

In [117]:
from pathlib import Path

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    CSVLoader,
    PyPDFLoader,
    WebBaseLoader,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [118]:

def load_and_split_documents(path_or_url):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    if path_or_url.startswith("http://") or \
       path_or_url.startswith("https://"):

        print("Loading web page...")

        loader = WebBaseLoader(path_or_url)

        documents = loader.load()

        chunks = splitter.split_documents(documents)

        return chunks

    path = Path(path_or_url)

    if path.is_dir():

        all_documents = []

        txt_loader = DirectoryLoader(
            str(path),
            glob="**/*.txt",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )

        all_documents.extend(txt_loader.load())

        csv_loader = DirectoryLoader(
            str(path),
            glob="**/*.csv",
            loader_cls=CSVLoader
        )

        all_documents.extend(csv_loader.load())

        pdf_loader = DirectoryLoader(
            str(path),
            glob="**/*.pdf",
            loader_cls=PyPDFLoader
        )

        all_documents.extend(pdf_loader.load())
        chunks = splitter.split_documents(all_documents)

        return chunks

    if path.is_file():
        if path.suffix == ".txt":
            loader = TextLoader(
                str(path),
                encoding="utf-8"
            )

        elif path.suffix == ".csv":
            loader = CSVLoader(str(path))
        elif path.suffix == ".pdf":
            loader = PyPDFLoader(str(path))
        else:
            raise ValueError(
                f"Unsupported file type: {path.suffix}"
            )

        documents = loader.load()
        chunks = splitter.split_documents(documents)
        return chunks

    raise ValueError(
        f"Path or URL does not exist: {path_or_url}"
    )

In [119]:
local_chunks = load_and_split_documents("data")

In [120]:
len(local_chunks)

10

In [121]:
if local_chunks:
    print("\nSample Chunk:")
    print(local_chunks[0].page_content)

    print("\nMetadata:")
    print(local_chunks[0].metadata)


Sample Chunk:
Personal Knowledge Assistant

LangChain is a framework for building applications powered by large language models.

A document loader is used to load information from different sources such as
text files, CSV files, PDF documents, websites, and directories.

Document loaders convert different types of data into LangChain Document objects.

Each Document generally contains:
1. page_content - the actual text
2. metadata - information about the source document

Metadata:
{'source': 'data\\notes.txt'}


In [122]:
url = "https://www.python.org/about/gettingstarted/"

web_chunks = load_and_split_documents(url)

print("Total chunks:", len(web_chunks))

if web_chunks:
    print("\nSample Web Chunk:")
    print(web_chunks[0].page_content[:500])

    print("\nMetadata:")
    print(web_chunks[0].metadata)

Loading web page...
Total chunks: 14

Sample Web Chunk:
Python For Beginners | Python.org




















Notice: This page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.





Skip to content


▼ Close
                


Python


PSF


Docs


PyPI


Jobs


Community



▲ The Python Network
                









Donate

≡ Menu


Search This Site

Metadata:
{'source': 'https://www.python.org/about/gettingstarted/', 'title': 'Python For Beginners | Python.org', 'description': 'The official home of the Python Programming Language', 'language': 'en'}


## TASK 12: Observation and Insights

1. Which loader is used for which data type?
- TextLoader for .txt
- CSVLoader for .csv
- PyPDFLoader for .pdf
- DirectoryLoader for Directory
- WebBaseLoader from Website

2. Best splitter for each type

Small text
- CharacterTextSplitter
- For small and simple text, character-based splitting is sufficient.

Large PDFs
- RecursiveCharacterTextSplitter
- It provides better splitting around paragraphs, sentences, and words while retaining PDF page metadata from the loader.

Web data
- RecursiveCharacterTextSplitter
- Web pages often contain large blocks of extracted text, so recursive splitting provides a reasonable general-purpose approach.

For highly structured web/HTML content, a structure-aware splitter can be preferable.

3. Why is chunk overlap important?

Chunk overlap prevents important information from being lost at chunk boundaries.